# Week 15: Images, convolution, and CNNs (capstone Experiment B begins, milestone M5)

**Experiment B:** pneumonia classification on **PneumoniaMNIST**, paediatric chest X-rays from MedMNIST v2 (Yang et al., 2023), derived from Kermany et al. (2018). It's licensed **CC BY 4.0** (verify this on medmnist.com and record it in `DATASETS.md`). The images are 28×28, so everything this week runs on your **CPU** in minutes.

> ⚠️ Experimental models in this project are **not clinically validated** and must never be used for diagnosis.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Images as arrays; convolution from scratch | 2.5 h |
| Day 2 | A baseline: logistic regression on pixels, AUROC with a CI | 2 h |
| Day 3 | A small CNN in PyTorch; class imbalance | 3 h |
| Day 4 | Move the code into `medlab/imaging/` with tests | 2 h |
| Review | Spaced recall | 0.5 h |

**Read:** CS231n notes, *Convolutional Neural Networks* (cs231n.github.io/convolutional-networks). **Install:** `pip install medmnist`.

### Milestone M5
- [ ] The logistic-regression and CNN **validation** AUROC with bootstrap 95% CIs, in `capstone/results/`.
- [ ] `medlab/imaging/{data,models,train}.py` with tests (shapes, one-batch overfit, determinism).
- [ ] A PneumoniaMNIST section in `DATASETS.md`, including what's known about patient-level splitting.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

In [ ]:
import random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from course.data import load_medmnist

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

data = load_medmnist("pneumoniamnist", size=28)
for split, (X, y) in data.items():
    print(f"{split:5s} {X.shape} {X.dtype}  pneumonia share {y.mean():.2f}")
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
Xtr, ytr = data["train"]
for ax, i in zip(axes.ravel(), np.r_[np.where(ytr == 0)[0][:6], np.where(ytr == 1)[0][:6]]):
    ax.imshow(Xtr[i], cmap="gray"); ax.set_title(["normal", "pneumonia"][ytr[i]]); ax.axis("off")
plt.show()

✍️ *The classes are imbalanced. What accuracy does 'always predict pneumonia' get? Why is AUROC a better headline metric here? And why do we report sensitivity and specificity at a chosen threshold as well?* (double-click to edit)



---
# Day 1: Convolution from scratch (2.5 h)

A greyscale image is a 2-D array of brightness values. A **convolution** slides a small **kernel** (for example 3×3) over the image and computes a weighted sum at every position. Deep-learning libraries actually compute *cross-correlation* (they don't flip the kernel), and we'll do the same.

Why convolutions suit images:
- **Local patterns:** edges and textures are local.
- **Weight sharing:** the same detector is used everywhere, so the parameter count is tiny compared with a fully connected layer.
- **Translation equivariance:** a shifted input gives a shifted output.

Stacking conv → ReLU → pool layers grows the **receptive field**, so later layers see larger regions (edges → textures → parts).

### Exercise 15.1: `conv2d`
A "valid" 2-D cross-correlation (no padding, stride 1) of a 2-D `image` with a 2-D `kernel`. The output shape is `(H − kh + 1, W − kw + 1)`. Two loops over output positions are fine. *(Bonus: a fully vectorised version using `np.lib.stride_tricks.sliding_window_view`.)*

In [ ]:
def conv2d(image, kernel):
    raise NotImplementedError

In [ ]:
img = Xtr[0].astype(np.float64) / 255
sobel_x = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=float)
ours = conv2d(img, sobel_x)
ref = F.conv2d(torch.tensor(img)[None, None], torch.tensor(sobel_x)[None, None])[0, 0].numpy()
assert ours.shape == (26, 26) and np.allclose(ours, ref)
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, (t, im) in zip(ax, [("image", img), ("vertical edges", ours), ("horizontal edges", conv2d(img, sobel_x.T))]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()
print("✅ 15.1 passed: your conv matches PyTorch")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`out[i, j] = np.sum(image[i:i + kh, j:j + kw] * kernel)`.

</details>


✍️ *How many parameters does a 3×3 conv layer with 1 input channel and 16 output channels have (including biases)? How many does a fully connected layer from 28×28 inputs to 16 outputs have?* (double-click to edit)



---
# Day 2: Baseline first: logistic regression on pixels (2 h)

**Always start with a simple baseline.** It tells you whether the deep model is worth its complexity, and it catches pipeline bugs early. Tune on **validation** and keep **test** untouched until Week 19.

### Exercise 15.2: Baseline with a bootstrap CI
Flatten the images and scale them to [0, 1]. Fit `LogisticRegression(C=…, max_iter=2000)` for C in {0.01, 0.1, 1}, and choose by **validation AUROC**. Then write `bootstrap_auc_ci(y, scores, n_boot=1000, seed=0)` → `(low, high)` (resample indices; skip resamples with only one class). Store `best_C`, `val_auc_lr` and `ci_lr`.

In [ ]:
from sklearn.linear_model import LogisticRegression
flat = lambda X: X.reshape(len(X), -1).astype(np.float32) / 255
Xva, yva = data["val"]

def bootstrap_auc_ci(y, scores, n_boot=1000, seed=0):
    raise NotImplementedError

best_C, val_auc_lr, ci_lr = None, None, None

In [ ]:
print(f"LogReg C={best_C}: val AUROC {val_auc_lr:.3f}, 95% CI [{ci_lr[0]:.3f}, {ci_lr[1]:.3f}]")
assert ci_lr[0] <= val_auc_lr <= ci_lr[1] and val_auc_lr > 0.7
print("✅ 15.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

In the bootstrap loop: `i = rng.integers(0, n, n)`; `if len(np.unique(y[i])) < 2: continue`; append `roc_auc_score(y[i], scores[i])`. Take the 2.5th and 97.5th percentiles.

</details>


---
# Day 3: A small CNN (3 h)

**The architecture:** `[conv 3×3, 16 channels → ReLU → maxpool 2] → [conv 3×3, 32 → ReLU → maxpool 2] → flatten → linear 64 → ReLU → dropout → linear 2`. Use `padding=1`, so the convolutions keep the spatial size and only pooling halves it: 28 → 14 → 7.

**Inputs** must have shape `(batch, channels, H, W)`, so add a channel dimension. **Normalise** with the *training* mean and std.

**Class imbalance:** pass `weight=` to `CrossEntropyLoss` (inverse class frequency) so that the minority class isn't ignored. It changes the decision threshold, not the ranking much. Think about why AUROC is fairly robust to this.

### Exercise 15.3: `SmallCNN` and training
Implement the model, then train for `EPOCHS` epochs with AdamW (lr 1e-3) and batch size 64. Keep the model with the best **validation AUROC** (use `copy.deepcopy(model.state_dict())`). Store `val_auc_cnn`.

In [ ]:
import copy
from torch.utils.data import TensorDataset, DataLoader
mean, std = Xtr.mean() / 255, Xtr.std() / 255
def to_tensor(X):
    return torch.tensor(((X.astype(np.float32) / 255 - mean) / std).astype(np.float32)).unsqueeze(1)     # (N, 1, 28, 28)
train_ds = TensorDataset(to_tensor(Xtr), torch.tensor(ytr))
val_X, val_y = to_tensor(Xva), yva
EPOCHS = 8

class SmallCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

val_auc_cnn = None

In [ ]:
assert SmallCNN()(torch.zeros(5, 1, 28, 28)).shape == (5, 2)
print(f"CNN val AUROC {val_auc_cnn:.3f} vs LogReg {val_auc_lr:.3f}")
assert val_auc_cnn > 0.85
print("✅ 15.3 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`self.features = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))`. The flattened size is `32 * 7 * 7`.

</details>

<details><summary>Hint 2</summary>

Class weights: `counts = np.bincount(ytr)`, `w = torch.tensor(len(ytr) / (2 * counts), dtype=torch.float32)`.

</details>

<details><summary>Hint 3</summary>

For AUROC you need probabilities: `torch.softmax(model(val_X), 1)[:, 1]` inside `model.eval()` and `torch.no_grad()`.

</details>


---
# Day 4: Package it: `capstone/medlab/imaging/` (2 h)

Move the notebook code into tested modules, so that Weeks 16–19 reuse it:
```
medlab/imaging/__init__.py
medlab/imaging/data.py     make_tensors(data, size_norm_from="train") -> dict of TensorDatasets; normalisation stats from train only
medlab/imaging/models.py   SmallCNN
medlab/imaging/train.py    set_seed, train_one_epoch, predict_proba(model, X, batch_size=512) -> np.ndarray of P(class 1)
                           fit(model, train_ds, val_X, val_y, epochs, lr, batch_size, seed, class_weights=True) -> (best_model, history)
tests/test_imaging.py      at least 4 tests: output shape; normalisation uses train stats only;
                           a single batch can be overfit; same seed → identical predictions
                           (call set_seed BEFORE constructing the model: weight initialisation is random too)
```
Load data with `medlab.data.load_medmnist`. You vendored `course/data.py` in Week 12, so `load_medmnist` is already in `medlab/data.py`. Tests should use tiny fake data, for example by setting `COURSE_FAKE_DATA=1` in the test with `monkeypatch.setenv`, or by building small random arrays directly.

In [ ]:
# Create the imaging package and tests in VS Code, then run the checker.

In [ ]:
import json
assert run_tests("tests/test_imaging.py") == 0
assert (CAP / "tests/test_imaging.py").read_text().count("def test_") >= 4
idata, imodels, itrain = fresh("medlab.imaging.data"), fresh("medlab.imaging.models"), fresh("medlab.imaging.train")
ds = idata.make_tensors(data)
itrain.set_seed(0)
model, hist = itrain.fit(imodels.SmallCNN(), ds["train"], ds["val"].tensors[0], data["val"][1], epochs=3, lr=1e-3, batch_size=64, seed=0)
p = itrain.predict_proba(model, ds["val"].tensors[0])
assert p.shape == (len(data["val"][1]),) and 0 <= p.min() and p.max() <= 1
auc = roc_auc_score(data["val"][1], p)
(CAP / "results").mkdir(exist_ok=True)
json.dump({"logreg": {"C": best_C, "val_auroc": val_auc_lr, "ci": ci_lr}, "small_cnn": {"val_auroc": auc}},
          open(CAP / "results" / "week15_imaging_val.json", "w"), indent=2)
print(f"package CNN val AUROC {auc:.3f}")
print("✅ 15.4 passed (milestone M5)")

---
# Review (30 min)
1. *(Week 5)* What does a matrix do geometrically? How is a conv layer a (very sparse) matrix?
2. *(Week 8)* What is AUROC, as a probability?
3. *(Week 8)* If one child contributed several X-rays, how could an image-level split leak? What does the PneumoniaMNIST documentation say about this? (Look it up for `DATASETS.md`.)
4. *(Week 14)* What are the 6 bugs from last week's script?
5. Why use weight sharing in images? What's the receptive field after two 3×3 conv + 2×2 pool blocks?
6. Why must the normalisation statistics come from the training set?

✍️ *Your answer:* (double-click to edit)

